<a href="https://colab.research.google.com/github/cols153/posture_checker/blob/main/notebooks/01_pose_estimation_mmpose_colab.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

⚠️ This notebook is intended to run on Google Colab (GPU recommended).

# Push-Up Pose Estimation with MMPose
The goal is to prepare clean keypoints data for push-up posture analysis.
Output used by notebook 02_feature_engineering_mmpose.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_NAME = 'posture_checker'
REPO_URL = 'https://github.com/cols153/posture_checker.git'

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_root = Path('/content') / REPO_NAME
    if not repo_root.exists():
        subprocess.run(['git','clone',REPO_URL,str(repo_root)], check=True)
    os.chdir(repo_root / 'notebooks')
    notebooks_dir = Path.cwd()
else:
    notebooks_dir = Path.cwd()

PROJECT_ROOT = notebooks_dir.parent if notebooks_dir.name=='notebooks' else notebooks_dir

DATA_DIR = PROJECT_ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
VIDEOS_DIR = RAW_DIR / 'videos'

CORRECT_DIR = VIDEOS_DIR / 'correct'
INCORRECT_DIR = VIDEOS_DIR / 'incorrect'

INTERIM_DIR = DATA_DIR / 'interim'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS_JSON_DIR = INTERIM_DIR / 'landmarks_json'
LANDMARKS_JSON_DIR.mkdir(parents=True, exist_ok=True)

LANDMARKS_CSV_PATH = INTERIM_DIR / 'pose_landmarks_pushups.csv'

print('Project root:', PROJECT_ROOT)
print('Videos dir:', VIDEOS_DIR)
print('CSV output:', LANDMARKS_CSV_PATH)


In [1]:
!nvidia-smi

Tue Mar 24 18:52:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Installation

In [2]:
!pip -q install -U condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [3]:
!conda create -y -n mmpose310 python=3.10
!conda run -n mmpose310 python -V

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/mmpose310

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.2.25-hbd8a1cb_0 
  icu                conda-forge/linux-64::icu-78.3-h33c6efd_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.7.4-hecca717_0 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0f

In [4]:
!conda run -n mmpose310 python -m pip install -q -U pip setuptools wheel
!conda run -n mmpose310 python -m pip install -q -U \
    torch==2.4.1+cu121 torchvision==0.19.1+cu121 torchaudio==2.4.1+cu121 \
    --index-url https://download.pytorch.org/whl/cu121
!conda run -n mmpose310 python -m pip install -q -U openmim
!conda run -n mmpose310 mim install mmengine
!conda run -n mmpose310 python -m pip install -q -U \
    "mmcv==2.2.0" \
    -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html
!conda run -n mmpose310 python -m pip install -q -U xtcocotools munkres mmdet
!conda run -n mmpose310 python -m pip install -q -U --force-reinstall numpy==1.26.4
!conda run -n mmpose310 python -m pip uninstall -y opencv-python opencv-python-headless
!conda run -n mmpose310 python -m pip install -q opencv-python==4.10.0.84
!conda run -n mmpose310 python -m pip install -q -U "setuptools<80"
!conda run -n mmpose310 bash -lc "sed -i \"s/mmcv_maximum_version = '2.2.0'/mmcv_maximum_version = '2.3.0'/\" /usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/__init__.py"
!conda run -n mmpose310 python -m pip install -q -U --no-deps mmpose
!conda run -n mmpose310 python -m pip install -q -U scipy matplotlib pillow tqdm packaging pyyaml requests einops json_tricks prettytable matplotlib-inline ipython pandas
!conda run -n mmpose310 env MPLBACKEND=Agg python -c "import mmpose, torch; print('mmpose', mmpose.__version__, 'cuda?', torch.cuda.is_available())"

Looking in links: https://download.openmmlab.com/mmcv/dist/cu121/torch2.4.0/index.html
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 78.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 124.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 150.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.5 MB/s  0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.

Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.

In [ ]:
from pathlib import Path

# Input video folders (stored in the GitHub repository)
CORRECT_DIR = PROJECT_ROOT / "data" / "raw" / "videos" / "correct"
INCORRECT_DIR = PROJECT_ROOT / "data" / "raw" / "videos" / "incorrect"

# Output folders for MMPose predictions and logs
KEYPOINTS_ROOT = PROJECT_ROOT / "data" / "processed" / "keypoints" / "mmpose"
POSE2D_DIR = KEYPOINTS_ROOT / "pose2d"
BODY3D_DIR = KEYPOINTS_ROOT / "body3d"
MANIFEST_DIR = KEYPOINTS_ROOT / "manifests"
LOGS_DIR = KEYPOINTS_ROOT / "logs"
TMP_DIR = KEYPOINTS_ROOT / "_tmp"

for p in [
    POSE2D_DIR / "correct",
    POSE2D_DIR / "incorrect",
    BODY3D_DIR / "correct",
    BODY3D_DIR / "incorrect",
    MANIFEST_DIR,
    LOGS_DIR,
    TMP_DIR,
]:
    p.mkdir(parents=True, exist_ok=True)

print("CORRECT_DIR   =", CORRECT_DIR)
print("INCORRECT_DIR =", INCORRECT_DIR)
print("POSE2D_DIR    =", POSE2D_DIR)
print("BODY3D_DIR    =", BODY3D_DIR)

Correct dir exists  : True
Incorrect dir exists: True


## 3. Clone MMPose

In [8]:
!rm -rf /content/mmpose
!git clone https://github.com/open-mmlab/mmpose.git /content/mmpose
%cd /content/mmpose

Cloning into '/content/mmpose'...
remote: Enumerating objects: 31282, done.
remote: Counting objects: 100% (2/2), done.
remote: Total 31282 (delta 1), reused 1 (delta 1), pack-reused 31280 (from 2)
Receiving objects: 100% (31282/31282), 54.81 MiB | 23.67 MiB/s, done.
Resolving deltas: 100% (21519/21519), done.
/content/mmpose


In [18]:
from pathlib import Path

patch_code = """
from pathlib import Path

site_pkg = Path('/usr/local/envs/mmpose310/lib/python3.10/site-packages')
vis_file = site_pkg / 'mmpose' / 'visualization' / 'local_visualizer_3d.py'
assert vis_file.exists(), f'File not found: {vis_file}'

text = vis_file.read_text(encoding='utf-8')
marker = 'import numpy as np\\n'
patch = '''import numpy as np

# --- Colab / Matplotlib compatibility patch ---
try:
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    if not hasattr(FigureCanvasAgg, "tostring_rgb"):
        def _tostring_rgb(self):
            buf = self.buffer_rgba()
            import numpy as _np
            rgba = _np.asarray(buf)
            rgb = _np.asarray(rgba[..., :3], order="C")
            return rgb.tobytes()
        FigureCanvasAgg.tostring_rgb = _tostring_rgb
except Exception:
    pass
'''

if 'Colab / Matplotlib compatibility patch' not in text:
    if marker not in text:
        raise RuntimeError("Could not find insertion point in local_visualizer_3d.py")
    text = text.replace(marker, marker + patch, 1)
    vis_file.write_text(text, encoding='utf-8')
    print('Patch applied:', vis_file)
else:
    print('Patch already present:', vis_file)
"

(Path('/content/patch_local_visualizer.py')
    .write_text(patch_code, encoding='utf-8'))
!python /content/patch_local_visualizer.py


1209

In [19]:
!conda run -n mmpose310 python /content/patch_local_visualizer_3d.py

Patch appliqué dans /usr/local/envs/mmpose310/lib/python3.10/site-packages/mmpose/visualization/local_visualizer_3d.py



## 4. Batch utilities 

In [20]:
from pathlib import Path
import subprocess
import shutil
import json
import pandas as pd
from datetime import datetime

VIDEO_EXTENSIONS = {'.mp4', '.mov', '.avi', '.mkv', '.mpeg', '.mpg', '.webm'}


def list_video_files(folder: Path):
    if not folder.exists():
        return []
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in VIDEO_EXTENSIONS])


def build_video_table():
    rows = []
    for label, folder in [('correct', CORRECT_DIR), ('incorrect', INCORRECT_DIR)]:
        for video_path in list_video_files(folder):
            rows.append({'label': label, 'video_name': video_path.name, 'video_stem': video_path.stem, 'video_path': str(video_path)})
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(['label', 'video_name']).reset_index(drop=True)
    return df


def run_shell(cmd_list, log_path: Path):
    printable = ' '.join(cmd_list)
    print('' + '=' * 100)
    print(printable)
    print('=' * 100)
    proc = subprocess.run(cmd_list, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_path.write_text(proc.stdout, encoding='utf-8')
    return proc.returncode


def find_json_files(root: Path):
    return sorted(root.rglob('*.json'))


def infer_expected_pose2d_json(video_path: Path, pred_dir: Path):
    candidate = pred_dir / f'{video_path.stem}.json'
    if candidate.exists():
        return candidate
    jsons = find_json_files(pred_dir)
    if len(jsons) == 1:
        return jsons[0]
    for js in jsons:
        if js.stem == video_path.stem:
            return js
    return None


def infer_body3d_json(tmp_root: Path):
    jsons = find_json_files(tmp_root)
    if not jsons:
        return None
    return sorted(jsons, key=lambda p: p.stat().st_size, reverse=True)[0]

In [21]:
video_df = build_video_table()
print('Total number of videos:', len(video_df))
if not video_df.empty:
    display(video_df.head())
    display(video_df['label'].value_counts())


Nombre total de vidéos : 100


,label,video_name,video_stem,video_path
0,correct,Copy of push up 1.mp4,Copy of push up 1,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
1,correct,Copy of push up 100.mp4,Copy of push up 100,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
2,correct,Copy of push up 101.mp4,Copy of push up 101,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
3,correct,Copy of push up 102.mp4,Copy of push up 102,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
4,correct,Copy of push up 113.mp4,Copy of push up 113,/content/drive/MyDrive/dne_mmpose/data/raw/vid...


,count
label,
correct,50
incorrect,50


## 5. Configuration

In [28]:
TEST_MODE = False
MAX_VIDEOS = 4
OVERWRITE_EXISTING = False
RUN_POSE2D = True
RUN_BODY3D = True
DEVICE = 'cuda'
POSE2D_ALIAS = 'human'

DET_CONFIG = 'demo/mmdetection_cfg/rtmdet_m_640-8xb32_coco-person.py'
DET_CKPT = 'https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth'
POSE2D_CONFIG = 'configs/body_2d_keypoint/rtmpose/body8/rtmpose-m_8xb256-420e_body8-256x192.py'
POSE2D_CKPT = 'https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/rtmpose-m_simcc-body7_pt-body7_420e-256x192-e48f03d0_20230504.pth'
POSE3D_CONFIG = 'configs/body_3d_keypoint/motionbert/h36m/motionbert_dstformer-243frm_8xb32-240e_h36m.py'
POSE3D_CKPT = 'https://download.openmmlab.com/mmpose/v1/body_3d_keypoint/pose_lift/h36m/motionbert_ft_h36m-d80af323_20230531.pth'

batch_df = video_df.copy()
if TEST_MODE and MAX_VIDEOS is not None and not batch_df.empty:
    batch_df = batch_df.groupby('label', group_keys=False).head(max(1, MAX_VIDEOS // 2)).reset_index(drop=True)

display(batch_df)

,label,video_name,video_stem,video_path
0,correct,Copy of push up 1.mp4,Copy of push up 1,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
1,correct,Copy of push up 100.mp4,Copy of push up 100,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
2,correct,Copy of push up 101.mp4,Copy of push up 101,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
3,correct,Copy of push up 102.mp4,Copy of push up 102,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
4,correct,Copy of push up 113.mp4,Copy of push up 113,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
...,...,...,...,...
95,incorrect,Copy of push up 55.mp4,Copy of push up 55,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
96,incorrect,Copy of push up 56.mp4,Copy of push up 56,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
97,incorrect,Copy of push up 57.mp4,Copy of push up 57,/content/drive/MyDrive/dne_mmpose/data/raw/vid...
98,incorrect,Copy of push up 58.mp4,Copy of push up 58,/content/drive/MyDrive/dne_mmpose/data/raw/vid...


## 6. Inference functions


In [29]:
def run_pose2d_for_video(video_path: Path, label: str):
    final_json = POSE2D_DIR / label / f'{video_path.stem}.json'
    log_path = LOGS_DIR / 'pose2d' / label / f'{video_path.stem}.log'
    if final_json.exists() and not OVERWRITE_EXISTING:
        return {'status': 'skipped_existing', 'json_path': str(final_json), 'log_path': str(log_path)}

    tmp_pred_dir = TMP_DIR / 'pose2d' / label / video_path.stem
    if tmp_pred_dir.exists():
        shutil.rmtree(tmp_pred_dir)
    tmp_pred_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        'conda', 'run', '-n', 'mmpose310', 'env', 'MPLBACKEND=Agg', 'python',
        'demo/inferencer_demo.py', str(video_path),
        '--pose2d', POSE2D_ALIAS,
        '--pred-out-dir', str(tmp_pred_dir),
        '--device', DEVICE,
    ]
    rc = run_shell(cmd, log_path)
    if rc != 0:
        return {'status': 'error', 'json_path': None, 'log_path': str(log_path)}

    raw_json = infer_expected_pose2d_json(video_path, tmp_pred_dir)
    if raw_json is None or not raw_json.exists():
        return {'status': 'json_not_found', 'json_path': None, 'log_path': str(log_path)}

    final_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(raw_json, final_json)
    return {'status': 'ok', 'json_path': str(final_json), 'log_path': str(log_path)}

In [30]:
def run_body3d_for_video(video_path: Path, label: str):
    final_json = BODY3D_DIR / label / f'{video_path.stem}.json'
    log_path = LOGS_DIR / 'body3d' / label / f'{video_path.stem}.log'
    if final_json.exists() and not OVERWRITE_EXISTING:
        return {'status': 'skipped_existing', 'json_path': str(final_json), 'log_path': str(log_path)}

    tmp_output_root = TMP_DIR / 'body3d' / label / video_path.stem
    if tmp_output_root.exists():
        shutil.rmtree(tmp_output_root)
    tmp_output_root.mkdir(parents=True, exist_ok=True)

    cmd = [
        'conda', 'run', '-n', 'mmpose310', 'env', 'MPLBACKEND=Agg', 'python',
        'demo/body3d_pose_lifter_demo.py',
        DET_CONFIG, DET_CKPT,
        POSE2D_CONFIG, POSE2D_CKPT,
        POSE3D_CONFIG, POSE3D_CKPT,
        '--input', str(video_path),
        '--output-root', str(tmp_output_root),
        '--save-predictions',
        '--device', DEVICE,
    ]
    rc = run_shell(cmd, log_path)
    if rc != 0:
        return {'status': 'error', 'json_path': None, 'log_path': str(log_path)}

    raw_json = infer_body3d_json(tmp_output_root)
    if raw_json is None or not raw_json.exists():
        return {'status': 'json_not_found', 'json_path': None, 'log_path': str(log_path)}

    final_json.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(raw_json, final_json)
    return {'status': 'ok', 'json_path': str(final_json), 'log_path': str(log_path)}

## 7. Run the batch  

In [31]:
results = []
started_at = datetime.now().isoformat(timespec='seconds')

for idx, row in batch_df.iterrows():
    video_path = Path(row['video_path'])
    label = row['label']
    print(f'\n[{idx + 1}/{len(batch_df)}] {label} :: {video_path.name}')

    record = {
        'label': label,
        'video_name': video_path.name,
        'video_stem': video_path.stem,
        'video_path': str(video_path),
        'pose2d_status': None,
        'pose2d_json_path': None,
        'pose2d_log_path': None,
        'body3d_status': None,
        'body3d_json_path': None,
        'body3d_log_path': None,
    }

    if RUN_POSE2D:
        r2d = run_pose2d_for_video(video_path, label)
        record['pose2d_status'] = r2d['status']
        record['pose2d_json_path'] = r2d['json_path']
        record['pose2d_log_path'] = r2d['log_path']

    if RUN_BODY3D:
        r3d = run_body3d_for_video(video_path, label)
        record['body3d_status'] = r3d['status']
        record['body3d_json_path'] = r3d['json_path']
        record['body3d_log_path'] = r3d['log_path']

    results.append(record)

finished_at = datetime.now().isoformat(timespec='seconds')
results_df = pd.DataFrame(results)
display(results_df)

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/models/backbones/csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/models/layers/se_layer.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/models/backbones/csp_darknet.py:118: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/envs/mmpose310/lib/python3.10/site-packages/mmdet/models/layers/se

,label,video_name,video_stem,video_path,pose2d_status,pose2d_json_path,pose2d_log_path,body3d_status,body3d_json_path,body3d_log_path
0,correct,Copy of push up 1.mp4,Copy of push up 1,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,skipped_existing,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,skipped_existing,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
1,correct,Copy of push up 100.mp4,Copy of push up 100,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,skipped_existing,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,skipped_existing,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
2,correct,Copy of push up 101.mp4,Copy of push up 101,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
3,correct,Copy of push up 102.mp4,Copy of push up 102,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
4,correct,Copy of push up 113.mp4,Copy of push up 113,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
...,...,...,...,...,...,...,...,...,...,...
95,incorrect,Copy of push up 55.mp4,Copy of push up 55,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
96,incorrect,Copy of push up 56.mp4,Copy of push up 56,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
97,incorrect,Copy of push up 57.mp4,Copy of push up 57,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...
98,incorrect,Copy of push up 58.mp4,Copy of push up 58,/content/drive/MyDrive/dne_mmpose/data/raw/vid...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...,ok,/content/drive/MyDrive/dne_mmpose/data/process...,/content/drive/MyDrive/dne_mmpose/data/process...


## 8. Save the manifest

In [32]:
run_name = 'test_run' if TEST_MODE else 'full_run'
manifest_path = MANIFEST_DIR / f'mmpose_pose_exports_{run_name}.csv'
summary_path = MANIFEST_DIR / f'mmpose_pose_exports_{run_name}_summary.json'

results_df.to_csv(manifest_path, index=False)
summary = {
    'started_at': started_at,
    'finished_at': finished_at,
    'test_mode': TEST_MODE,
    'n_videos': int(len(results_df)),
    'pose2d_status_counts': results_df['pose2d_status'].value_counts(dropna=False).to_dict() if 'pose2d_status' in results_df else {},
    'body3d_status_counts': results_df['body3d_status'].value_counts(dropna=False).to_dict() if 'body3d_status' in results_df else {},
    'manifest_path': str(manifest_path),
}
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print('Manifest CSV:', manifest_path)
print('Summary JSON:', summary_path)
print(json.dumps(summary, indent=2))

Manifest CSV: /content/drive/MyDrive/dne_mmpose/data/processed/keypoints/mmpose/manifests/mmpose_pose_exports_full_run.csv
Summary JSON: /content/drive/MyDrive/dne_mmpose/data/processed/keypoints/mmpose/manifests/mmpose_pose_exports_full_run_summary.json
{
  "started_at": "2026-03-24T19:12:19",
  "finished_at": "2026-03-24T20:34:28",
  "test_mode": false,
  "n_videos": 100,
  "pose2d_status_counts": {
    "ok": 96,
    "skipped_existing": 4
  },
  "body3d_status_counts": {
    "ok": 96,
    "skipped_existing": 4
  },
  "manifest_path": "/content/drive/MyDrive/dne_mmpose/data/processed/keypoints/mmpose/manifests/mmpose_pose_exports_full_run.csv"
}


In [33]:
pose2d_examples = sorted(POSE2D_DIR.rglob('*.json'))
body3d_examples = sorted(BODY3D_DIR.rglob('*.json'))
print('Nb JSON 2D:', len(pose2d_examples))
print('Nb JSON 3D:', len(body3d_examples))

Nb JSON 2D: 100
Nb JSON 3D: 100


In [34]:
import os

BASE_DIR = "/content/drive/MyDrive/dne_mmpose/data/processed/keypoints/mmpose"

POSE2D_DIR = os.path.join(BASE_DIR, "_tmp", "pose2d")
POSE3D_DIR = os.path.join(BASE_DIR, "_tmp", "body3d")

def check_outputs(base_dir, name, expected=50):
    print(f"\n🔍 Checking {name}...")

    total = 0

    for label in ["correct", "incorrect"]:
        label_dir = os.path.join(base_dir, label)

        if not os.path.exists(label_dir):
            print(f"❌ Missing folder: {label_dir}")
            continue

        videos = os.listdir(label_dir)
        count = len(videos)
        total += count

        print(f"  {label}: {count} videos")

        # Check empty folders
        for v in videos:
            v_path = os.path.join(label_dir, v)
            if len(os.listdir(v_path)) == 0:
                print(f"  ⚠️ Empty folder: {v}")

    print(f"➡️ Total: {total} videos")

    if total >= expected * 2:
        print("✅ Looks good!")
    else:
        print("⚠️ Missing some videos")

# Run checksun checks
check_outputs(POSE2D_DIR, "Pose2D")
check_outputs(POSE3D_DIR, "Body3D")


🔍 Checking Pose2D...
  correct: 50 videos
  incorrect: 50 videos
➡️ Total: 100 videos
✅ Looks good!

🔍 Checking Body3D...
  correct: 50 videos
  incorrect: 50 videos
➡️ Total: 100 videos
✅ Looks good!


In [37]:
import os

BASE_DIR = "/content/drive/MyDrive/dne_mmpose/data/processed/keypoints/mmpose"
POSE2D_DIR = os.path.join(BASE_DIR, "pose2d")
POSE3D_DIR = os.path.join(BASE_DIR, "body3d")

def check_final_exports(base_dir, name, expected_total=100):
    print(f"\n🔍 Checking final exports: {name}")
    total = 0
    missing = 0

    for label in ["correct", "incorrect"]:
        label_dir = os.path.join(base_dir, label)

        if not os.path.exists(label_dir):
            print(f"❌ Missing folder: {label_dir}")
            continue

        files = [f for f in os.listdir(label_dir) if f.endswith(".json")]
        total += len(files)

        print(f"  {label}: {len(files)} json files")

        for f in files[:3]:
            print(f"    - {f}")

    print(f"➡️ Total: {total} json files")

    if total == expected_total:
        print("✅ Final exports look good!")
    else:
        print("⚠️ Unexpected total number of JSON files")

check_final_exports(POSE2D_DIR, "Pose2D")
check_final_exports(POSE3D_DIR, "Body3D")


🔍 Checking final exports: Pose2D
  correct: 50 json files
    - Copy of push up 1.json
    - Copy of push up 100.json
    - Copy of push up 101.json
  incorrect: 50 json files
    - 1.json
    - 10.json
    - 11.json
➡️ Total: 100 json files
✅ Final exports look good!

🔍 Checking final exports: Body3D
  correct: 50 json files
    - Copy of push up 1.json
    - Copy of push up 100.json
    - Copy of push up 101.json
  incorrect: 50 json files
    - 1.json
    - 10.json
    - 11.json
➡️ Total: 100 json files
✅ Final exports look good!


## 9. Full batch

Once the test run looks good:
1. set `TEST_MODE = False`
2. rerun the setup cells and then the batch
3. download or copy the generated JSON files for the next notebooks
